Install All Packages

In [9]:
!pip install -q groq langchain langchain-groq langchain-huggingface \
             langchain-text-splitters langchain-core langchain-community \
             sentence-transformers faiss-cpu youtube-transcript-api

**Set Groq API Key**

In [ ]:
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

**Indexing: Document Ingestion**

In [11]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_core.documents import Document

VIDEO_ID = "5MuIMqhT8DM"  # replace with your video ID

ytt     = YouTubeTranscriptApi()
fetched = ytt.fetch(VIDEO_ID)

print("=" * 50)
print("INDEXING STEP 1: DOCUMENT INGESTION")
print("=" * 50)

# Show transcript list
print("\n📄 Transcript List (first 5 segments):")
for t in fetched[:5]:
    print(f"  [{int(t.start)}s] {t.text}")

# Full transcript text
full_text = " ".join([t.text for t in fetched])
documents = [Document(page_content=full_text, metadata={"source": VIDEO_ID})]

print(f"\n✅ Total characters : {len(full_text)}")
print(f"✅ Total documents  : {len(documents)}")
print(f"\nPreview:\n{full_text[:400]}")
# Show ALL transcript segments
print("\n📄 Full Transcript List:")
for t in fetched:
    print(f"  [{int(t.start)}s] {t.text}")

INDEXING STEP 1: DOCUMENT INGESTION

📄 Transcript List (first 5 segments):
  [0s] Thank you very much.
  [2s] Well, I would like
to start with testicles.
  [6s] (Laughter)
  [9s] Men who sleep five hours a night
  [11s] have significantly smaller testicles
than those who sleep seven hours or more.

✅ Total characters : 15024
✅ Total documents  : 1

Preview:
Thank you very much. Well, I would like
to start with testicles. (Laughter) Men who sleep five hours a night have significantly smaller testicles
than those who sleep seven hours or more. (Laughter) In addition, men who routinely sleep
just four to five hours a night will have a level of testosterone which is that of someone
10 years their senior. So a lack of sleep
will age a man by a decade in t

📄 Full Transcript List:
  [0s] Thank you very much.
  [2s] Well, I would like
to start with testicles.
  [6s] (Laughter)
  [9s] Men who sleep five hours a night
  [11s] have significantly smaller testicles
than those who sleep seven hours

**Indexing Part 2: Text Splitting**

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("=" * 50)
print("INDEXING STEP 2: TEXT SPLITTING (Time-Based)")
print("=" * 50)

chunk_duration = 15  # seconds per chunk
chunks         = []
current_text   = []
current_start  = None
current_end    = None

for t in fetched:
    start = int(t.start)
    end   = int(t.start + t.duration)

    if current_start is None:
        current_start = start

    current_text.append(t.text)
    current_end = end

    if (current_end - current_start) >= chunk_duration:
        start_fmt = f"{current_start // 60:02d}:{current_start % 60:02d}"
        end_fmt   = f"{current_end   // 60:02d}:{current_end   % 60:02d}"

        chunks.append(Document(
            page_content=" ".join(current_text),
            metadata={
                "source"    : VIDEO_ID,
                "timestamp" : f"[{start_fmt} - {end_fmt}]",
                "start_sec" : current_start,
                "end_sec"   : current_end
            }
        ))
        current_text  = []
        current_start = None

# Save remaining text
if current_text:
    start_fmt = f"{current_start // 60:02d}:{current_start % 60:02d}"
    end_fmt   = f"{current_end   // 60:02d}:{current_end   % 60:02d}"
    chunks.append(Document(
        page_content=" ".join(current_text),
        metadata={
            "source"    : VIDEO_ID,
            "timestamp" : f"[{start_fmt} - {end_fmt}]",
            "start_sec" : current_start,
            "end_sec"   : current_end
        }
    ))

print(f"✅ Total chunks created: {len(chunks)}")

INDEXING STEP 2: TEXT SPLITTING (Time-Based)
✅ Total chunks created: 68


**Create Embeddings + Vector Store**

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("=" * 50)
print("INDEXING STEP 3 & 4: EMBEDDING + VECTOR STORE")
print("=" * 50)

print("\n⏳ Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Embedding model loaded!")

print("\n⏳ Generating embeddings and storing in FAISS...")
vectorstore = FAISS.from_documents(chunks, embeddings)
print("✅ Embeddings stored in vector store!")
print(f"✅ Total vectors indexed: {len(chunks)}")

/tmp/ipykernel_2622/3374951852.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


INDEXING STEP 3 & 4: EMBEDDING + VECTOR STORE

⏳ Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!

⏳ Generating embeddings and storing in FAISS...
✅ Embeddings stored in vector store!
✅ Total vectors indexed: 68


**Augmentation**

In [16]:
from langchain_core.prompts import ChatPromptTemplate

print("=" * 50)
print("STEP 3: AUGMENTATION")
print("=" * 50)

# Question
question       = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)

print(f"🔍 Question:\n{question}\n")

# Retrieved docs
print(f"📄 Retrieved Docs ({len(retrieved_docs)} chunks):")
for i, doc in enumerate(retrieved_docs):
    print(f"\n  Chunk {i+1} {doc.metadata['timestamp']}")
    print(f"  {doc.page_content}")

# Context text
context = "\n\n".join([
    f"{doc.metadata['timestamp']}\n{doc.page_content}"
    for doc in retrieved_docs
])
print(f"\n📋 Context Text:\n{context}")

# Prompt template
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant that answers questions about a YouTube video.
Answer ONLY based on the context below.
If the answer is not in the context, say "I could not find that in this video."

Context:
{context}

Question:
{question}

Answer:
""")

# Final prompt (show how it looks)
final_prompt = prompt.format(context=context, question=question)
print(f"\n📝 Final Prompt Sent to LLM:\n")
print(final_prompt)

STEP 3: AUGMENTATION
🔍 Question:
is the topic of nuclear fusion discussed in this video? if yes then what was discussed

📄 Retrieved Docs (3 chunks):

  Chunk 1 [11:09 - 11:25]
  Currently, that list includes
cancer of the bowel, cancer of the prostate
and cancer of the breast. In fact, the link between a lack of sleep
and cancer is now so strong that the World Health Organization

  Chunk 2 [02:48 - 03:05]
  And when you put
those two groups head to head, what you find is a quite significant,
40-percent deficit in the ability of the brain
to make new memories without sleep. I think this should be concerning, considering what we know
is happening to sleep

  Chunk 3 [02:32 - 02:47]
  and we're going to have them
try and learn a whole list of new facts as we're taking snapshots
of brain activity. And then we're going to test them to see how effective
that learning has been. And that's what you're looking at
here on the vertical axis.

📋 Context Text:
[11:09 - 11:25]
Currently, that list

**Generation**

In [17]:
from langchain_groq import ChatGroq

print("=" * 50)
print("STEP 4: GENERATION")
print("=" * 50)

llm      = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
response = llm.invoke(final_prompt)

print(f"🤖 Answer:\n{response.content}")

STEP 4: GENERATION


AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

**Building a chain**

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("=" * 50)
print("STEP 5: BUILDING THE CHAIN")
print("=" * 50)

def format_docs(docs):
    return "\n\n".join([
        f"{doc.metadata['timestamp']}\n{doc.page_content}"
        for doc in docs
    ])

rag_chain = (
    {
        "context" : retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG Chain built!\n")
print("🎬 Chat with your video! Type 'exit' to stop.\n")

while True:
    question = input("You: ")
    if question.lower() == "exit":
        print("Bye!")
        break

    answer     = rag_chain.invoke(question)
    rel_docs   = retriever.invoke(question)
    timestamps = " | ".join([doc.metadata["timestamp"] for doc in rel_docs])

    print(f"\n🤖 Bot: {answer}")
    print(f"📍 Timestamp: {timestamps}\n")

STEP 5: BUILDING THE CHAIN
✅ RAG Chain built!

🎬 Chat with your video! Type 'exit' to stop.

